# BRT 360 - train your own telecom CRM model (free Colab GPU)

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`. Then `Runtime → Run all`.

What happens (about 45-60 minutes, all inside your Google Colab session):
1. installs the training tools,
2. asks you to **upload `brt360_training_data.jsonl`** (from Admin → AI Settings → Download training data),
3. fine-tunes **Qwen 2.5 3B** - the same base your server runs - on it,
4. checks it on examples it never saw,
5. exports **`brt360-telecom.gguf`** (~1.9 GB) and saves it to your Google Drive (and offers a direct download).

Then on the server: `bash /opt/brt360/deploy/local_llm.sh import /tmp/brt360-telecom.gguf`.

*The file contains phrases your users typed. It is processed in your own Colab session and saved to your own Drive.*

In [ ]:
# 1. Check the GPU (must show a Tesla T4 or better)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
# 2. Install the training tools (3-5 minutes)
!pip install unsloth

In [ ]:
# 3. Upload brt360_training_data.jsonl
from google.colab import files
uploaded = files.upload()
DATA = next(name for name in uploaded if name.endswith(".jsonl"))
import json, random
rows = [json.loads(l) for l in open(DATA, encoding="utf-8") if l.strip()]
random.Random(7).shuffle(rows)
n_eval = max(5, int(len(rows) * 0.05))
eval_rows, train_rows = rows[:n_eval], rows[n_eval:]
with open("brt360-telecom-holdout.jsonl", "w", encoding="utf-8") as fh:
    fh.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in eval_rows)
print(f"{len(train_rows)} examples to learn from, {len(eval_rows)} kept aside to test")

In [ ]:
# 4. Settings - change only if you know why
BASE = "unsloth/Qwen2.5-3B-Instruct"   # must match the server's base (qwen2.5:3b-instruct)
MAX_SEQ = 4096                         # the CRM prompt is ~2,400 tokens
EPOCHS = 2
LR = 2e-4
RANK = 16

In [ ]:
# 5. Load the base model + attach LoRA adapters (trains ~1% of the weights)
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template, train_on_responses_only

model, tokenizer = FastLanguageModel.from_pretrained(model_name=BASE, max_seq_length=MAX_SEQ, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=RANK, lora_alpha=RANK, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=7,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

In [ ]:
# 6. Train (about 30-45 minutes on a T4)
import inspect
from datasets import Dataset
from trl import SFTConfig, SFTTrainer

def to_text(batch):
    return {"text": [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in batch["messages"]]}

train_ds = Dataset.from_list(train_rows).map(to_text, batched=True, remove_columns=["messages"])
eval_ds = Dataset.from_list(eval_rows).map(to_text, batched=True, remove_columns=["messages"])

def accepted(cls, **kw):  # keyword names moved between TRL / transformers versions
    params = inspect.signature(cls.__init__).parameters
    return {k: v for k, v in kw.items() if k in params}

seq_key = "max_length" if "max_length" in inspect.signature(SFTConfig.__init__).parameters else "max_seq_length"
eval_key = "eval_strategy" if "eval_strategy" in inspect.signature(SFTConfig.__init__).parameters else "evaluation_strategy"
config = SFTConfig(**accepted(SFTConfig, **{
    "dataset_text_field": "text", seq_key: MAX_SEQ, "packing": False,
    "per_device_train_batch_size": 1, "gradient_accumulation_steps": 8, "num_train_epochs": EPOCHS,
    "learning_rate": LR, "lr_scheduler_type": "cosine", "warmup_ratio": 0.03, "weight_decay": 0.01,
    "optim": "adamw_8bit", "bf16": is_bfloat16_supported(), "fp16": not is_bfloat16_supported(),
    "logging_steps": 10, eval_key: "epoch", "save_strategy": "no", "output_dir": "checkpoints", "seed": 7, "report_to": "none",
}))
tok_key = "processing_class" if "processing_class" in inspect.signature(SFTTrainer.__init__).parameters else "tokenizer"
trainer = SFTTrainer(model=model, train_dataset=train_ds, eval_dataset=eval_ds, args=config, **{tok_key: tokenizer})
# learn the answers only - the long CRM instructions are context, not text to imitate
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
stats = trainer.train()
print(f"Done in {stats.metrics['train_runtime'] / 60:.0f} min - final training loss {stats.metrics['train_loss']:.3f}")

In [ ]:
# 7. Test on the examples it never saw
FastLanguageModel.for_inference(model)
ok = total = 0
for row in eval_rows:
    msgs = row["messages"]
    want = json.loads(msgs[-1]["content"])
    if "intent_type" not in want:
        continue
    ids = tokenizer.apply_chat_template(msgs[:-1], add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=300, do_sample=False)
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    try:
        got = json.loads(text[text.find("{"): text.rfind("}") + 1]).get("intent_type")
    except Exception:
        got = None
    total += 1; ok += got == want["intent_type"]
    print(("OK  " if got == want["intent_type"] else "MISS"), msgs[1]["content"][:70], "->", got)
print(f"\nIntent accuracy on unseen requests: {ok}/{total}")

In [ ]:
# 8. Export for the server (GGUF, 4-bit) - 5-10 minutes
model.save_pretrained_gguf("brt360-telecom-gguf", tokenizer, quantization_method="q4_k_m")
import glob, shutil
gguf = max(glob.glob("brt360-telecom-gguf*/*.gguf") + glob.glob("*.gguf"), key=lambda p: __import__("os").path.getsize(p))
shutil.copy(gguf, "brt360-telecom.gguf")
print("Exported:", gguf)

In [ ]:
# 9. Save to your Google Drive (reliable for a 2 GB file) and offer direct downloads
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs("/content/drive/MyDrive/BRT360", exist_ok=True)
for f in ("brt360-telecom.gguf", "brt360-telecom-holdout.jsonl"):
    shutil.copy(f, f"/content/drive/MyDrive/BRT360/{f}")
print("Saved to Google Drive -> MyDrive/BRT360/")
files.download("brt360-telecom-holdout.jsonl")
files.download("brt360-telecom.gguf")

## Install it on the BRT 360 server

From your PC (after downloading `brt360-telecom.gguf` and `brt360-telecom-holdout.jsonl` from Drive):
```
scp brt360-telecom.gguf brt360-telecom-holdout.jsonl root@187.127.132.149:/tmp/
```
On the server:
```
bash /opt/brt360/deploy/local_llm.sh import /tmp/brt360-telecom.gguf
python3 /opt/brt360/deploy/llm/evaluate.py --data /tmp/brt360-telecom-holdout.jsonl
python3 /opt/brt360/deploy/llm/evaluate.py --data /tmp/brt360-telecom-holdout.jsonl --model brt360-telecom-previous
```
If the new model scores lower, roll back: `ollama cp brt360-telecom-previous brt360-telecom`.